<a href="https://colab.research.google.com/github/Muhozgu/chlorobiota/blob/main/chlorobiota.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL, use_fast=False)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("Ready!")

In [ ]:
from PIL import Image

image = Image.open("/content/pic.png").convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": """You are a botanist. Identify this plant and respond in exactly this format:

Common name:
Family name:
Scientific name:
Description: """},
        ],
    },
]

text_prompt = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = processor(
    text=[text_prompt],
    images=[image],
    return_tensors="pt",
).to(model.device)

print(f"Tokens: {inputs['input_ids'].shape[1]}")

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        repetition_penalty=1.05,
    )

trimmed = generated_ids[0][inputs['input_ids'].shape[1]:]
result = processor.decode(trimmed, skip_special_tokens=True)
print(result)